In [1]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import session
import h5py
import pickle
from scipy import stats

In [2]:
# set the main data directory (this needs to be changed by each user)
maindir = '/Volumes/ExternalSSD/Credit_assignement'

In [3]:
# get all the sessions to analyze
mouse = 'M1'
layers = 'L1' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M1'
layers = 'L23' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M2'
layers = 'L1' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M2'
layers = 'L5' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M3'
layers = 'L1' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M3'
layers = 'L23' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M4'
layers = 'L1' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()

mouse = 'M4'
layers = 'L5' #'L1' 'L23' or 'L5'
sessionfile = open(mouse+'_'+layers+'_Sessions.txt','r')
allsessions = allsessions + [sess.rstrip('\n') for sess in sessionfile.readlines()]
sessionfile.close()


In [4]:
print(allsessions)

['720520553', '722126561', '723323411', '723322122', '721975393', '721032982', '720519615', '716711420', '715244457', '717030161', '716425232', '724422062', '725010342', '726844248', '727683236', '727680211', '725009252', '724421207', '726837871', '718742560', '722188453', '719034388', '718579351']


In [5]:
def get_dff_excess(minsess,maxsess,kappa,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict):
    sts = [] #surprise chunks
    nts = [] #no surprise chunks
    nbas = [] #no surp baseline
    sbas = [] #surp baseline
    
    for snum in range(minsess,maxsess):
        S = allsessions[snum]
        print(S)
        ssegs= sdict[S].gabors.get_segs_by_criteria(gaborframe=3,surp=mysurp,stimPar2=kappa,remconsec=True)
        sframes = gabordict[S].get_2pframes_by_seg(ssegs[0])
        straces = sdict[S].get_roi_segments(sframes,padding=pad)
        sts.append(straces[:,pad[0]:pad[0]+numframes,0:numtotake])#E responses
        sbas.append(straces[:,0:pad[0],0:numtotake])
    
        nsegs= sdict[S].gabors.get_segs_by_criteria(gaborframe=3,surp=0,stimPar2=kappa,remconsec=False)
        nframes = gabordict[S].get_2pframes_by_seg(nsegs[0])
        ntraces = sdict[S].get_roi_segments(nframes,padding=pad)
        nts.append(ntraces[:,pad[0]:pad[0]+numframes,0:500])#D responses
        nbas.append(ntraces[:,0:pad[0],0:500])
    
    flat_sts  = np.array([item for sublist in sts for item in sublist])
    flat_nts = np.array([item for sublist in nts for item in sublist])

    flat_nbas = np.array([item for sublist in nbas for item in sublist])
    flat_sbas = np.array([item for sublist in sbas for item in sublist])
        
    dff_sts = np.divide(np.nanmean(flat_sts,axis=1) - np.nanmean(flat_sbas,axis=1),np.nanmean(flat_sbas,axis=1))
    dff_nts = np.divide(np.nanmean(flat_nts,axis=1) - np.nanmean(flat_nbas,axis=1),np.nanmean(flat_nbas,axis=1))
        
    excess_activity = np.nanmean(dff_sts,axis=1)-np.nanmean(dff_nts,axis=1)
    excess_activity[abs(excess_activity)>1E6] = 'nan'
    return excess_activity

In [ ]:
# create a dictionary with Session objects prepared for analysis
sdict = {}
gabordict = {}
for sess in allsessions:                       # remove the :1 to get all sessions ready
    print("\nCreating session {}...".format(sess))
    sdict[sess] = session.Session(maindir,sess)    # creates a session object to work with
    sdict[sess].extract_info()  
    gabordict[sess] = session.Stim(sdict[sess],1,'gabors')
    print("finished session {}.".format(sess))
    


Creating session 720520553...
Loading stimulus dictionary...
Loading alignment dataframe...
NOTE: Stimulus alignment pickle already exists in /Volumes/ExternalSSD/Credit_assignement/ophys_session_720520553
Loading running data...
Loading ROI trace info...




finished session 720520553.

Creating session 722126561...
Loading stimulus dictionary...
Loading alignment dataframe...
NOTE: Stimulus alignment pickle already exists in /Volumes/ExternalSSD/Credit_assignement/ophys_session_722126561
Loading running data...
Loading ROI trace info...




finished session 722126561.

Creating session 723323411...
Loading stimulus dictionary...
Loading alignment dataframe...
NOTE: Stimulus alignment pickle already exists in /Volumes/ExternalSSD/Credit_assignement/ophys_session_723323411
Loading running data...
Loading ROI trace info...




finished session 723323411.

Creating session 723322122...
Loading stimulus dictionary...
Loading alignment dataframe...
NOTE: Stimulus alignment pickle already 

In [ ]:
numframes = 30 #number of frames to take df/f over
numtotake = 30 #number of surprise trials to take
pad=(180,40)
mysurp = 1

print(mysurp)
print('M1 L1')
M1L1_diffs_surp_k4 = get_dff_excess(0,3,4,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
print('M1 L23')
M1L23_diffs_surp_k4 = get_dff_excess(3,7,4,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
print('M3 L1')
M3L1_diffs_surp_k16 = get_dff_excess(11,15,16,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
print('M3 L23')
M3L23_diffs_surp_k16 = get_dff_excess(15,19,16,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)

In [ ]:
means = [np.mean(M1L1_diffs_surp_k4),np.mean(M3L1_diffs_surp_k16)]
stds = [np.std(M1L1_diffs_surp_k4)/np.sqrt(len(M1L1_diffs_surp_k4)),np.std(M3L1_diffs_surp_k16)/np.sqrt(len(M3L1_diffs_surp_k16))]
plt.bar([1,2],means,yerr=stds)
plt.xticks([1,2],['K 4 (M1 L1)','K 16 (M3 L1)'])
plt.ylabel('Diff in df/f for E vs preceding D (surprise)')
plt.title('Kappa Effect on E vs D ROI responses, Layer 1')
plt.savefig('Layer1_Mouse1and3_EvsD_GaborResponses_KappaBarPlot_Delta.png')

In [ ]:
means = [np.mean(M1L23_diffs_surp_k4),np.mean(M3L23_diffs_surp_k16)]
stds = [np.std(M1L23_diffs_surp_k4)/np.sqrt(len(M1L23_diffs_surp_k4)),np.std(M3L23_diffs_surp_k16)/np.sqrt(len(M3L23_diffs_nosurp_k16))]
plt.bar([1,2],means,yerr=stds)
plt.xticks([1,2],['K 4 Surp (M1 L23)','K 16 Surp (M3 L23)'])
plt.ylabel('Diff in df/f for E vs preceding D (surprise)')
plt.title('Kappa Effect on E vs D ROI responses, Layer 23')
plt.savefig('Layer23_Mouse1and3_EvsD_GaborResponses_KappaBarPlot_Delta.png')

In [ ]:
print(mysurp)

print('M2 L1')
M2L1_diffs_surp_k4 = get_dff_excess(7,9,4,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
M2L1_diffs_surp_k16 = get_dff_excess(7,9,16,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)

print('M2 L5')
M2L5_diffs_surp_k4 = get_dff_excess(9,11,4,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
M2L5_diffs_surp_k16 = get_dff_excess(9,11,16,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)

print('M4 L1')
M4L1_diffs_surp_k4 = get_dff_excess(19,21,4,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
M4L1_diffs_surp_k16 = get_dff_excess(19,21,16,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)

print('M4 L5')
M4L5_diffs_surp_k4 = get_dff_excess(21,23,4,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)
M4L5_diffs_surp_k16 = get_dff_excess(21,23,16,mysurp,numframes,numtotake,pad,allsessions,sdict,gabordict)


In [ ]:
means = [np.nanmean(M2L5_diffs_surp_k4),np.nanmean(M2L5_diffs_surp_k16),np.nanmean(M4L5_diffs_surp_k4),np.nanmean(M4L5_diffs_surp_k16)]
stds = [np.nanstd(M2L5_diffs_surp_k4)/np.sqrt(len(M2L5_diffs_surp_k4)),np.nanstd(M2L5_diffs_surp_k16)/np.sqrt(len(M2L5_diffs_surp_k16)),np.nanstd(M4L5_diffs_surp_k4)/np.sqrt(len(M4L5_diffs_surp_k4)),np.nanstd(M4L5_diffs_surp_k16)/np.sqrt(len(M4L5_diffs_surp_k16))]
plt.bar([1,2,3,4],means,yerr=stds)
plt.xticks([1,2,3,4],['K 4 (M2 L5)','K 16 (M2 L5)','K 4 (M4 L5)','K 16 (M4 L5)' ])
plt.ylabel('Diff in df/f for D vs preceding D (NO surprise)')
plt.title('Kappa Effect on D vs D ROI responses, Layer 5')
plt.savefig('Layer5_Mouse2and4_DvsD_GaborResponses_KappaBarPlot_Delta.png')

In [ ]:
means = [np.nanmean(M2L1_diffs_surp_k4),np.nanmean(M2L1_diffs_surp_k16),np.nanmean(M4L1_diffs_surp_k4),np.nanmean(M4L1_diffs_surp_k16)]
stds = [np.nanstd(M2L1_diffs_surp_k4)/np.sqrt(len(M2L1_diffs_surp_k4)),np.nanstd(M2L1_diffs_surp_k16)/np.sqrt(len(M2L1_diffs_surp_k16)),np.nanstd(M4L1_diffs_surp_k4)/np.sqrt(len(M4L1_diffs_surp_k4)),np.nanstd(M4L1_diffs_surp_k16)/np.sqrt(len(M4L1_diffs_surp_k16))]
plt.bar([1,2,3,4],means,yerr=stds)
plt.xticks([1,2,3,4],['K 4 (M2 L1)','K 16 (M2 L1)','K 4 (M4 L1)','K 16 (M4 L1)' ])
plt.ylabel('Diff in df/f for D vs preceding D (NO surprise)')
plt.title('Kappa Effect on D vs D ROI responses, Layer 1')
plt.savefig('Layer1_Mouse2and4_DvsD_GaborResponses_KappaBarPlot_Delta.png')